# RTE teacher / baseline: DeBERTa-v3-base on Colab

Runs the **same code and pinned packages** as the local venv. Produces:
1. `outputs/baseline_deberta_v3_base/`: baseline metrics (5 seeds), dev predictions, best-val checkpoint
2. `outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy`: out-of-fold soft labels for distillation

Outputs are written to Google Drive (step 1b), so a disconnect doesn't lose them; re-run any training cell to resume.

Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100 is faster).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

## 1. Get the code
Skips the clone if the repo is already here.

In [ ]:
import os
REPO_URL = "https://github.com/ihemanthm/rte-challenge.git"  # private repo: upload a zip instead

%cd /content
if not os.path.isdir("/content/rte-challenge"):
    if REPO_URL:
        !git clone -q $REPO_URL rte-challenge
    else:
        from google.colab import files
        up = files.upload()  # choose rte-challenge.zip
        !unzip -q -o {next(iter(up))} -d /content
%cd /content/rte-challenge
!git log --oneline -1 2>/dev/null || true

## 1b. Keep outputs on Google Drive
`outputs/` becomes a link to `MyDrive/rte-challenge/outputs`, so an expired runtime loses nothing. Training saves each finished seed and fold right away, and **re-running a training cell resumes**: finished seeds and folds are skipped.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/rte-challenge/outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)
%cd /content/rte-challenge
if os.path.isdir("outputs") and not os.path.islink("outputs"):
    !cp -rn outputs/. "$DRIVE_OUT"/ && rm -rf outputs
if not os.path.islink("outputs"):
    os.symlink(DRIVE_OUT, "outputs")
!ls -la outputs/ outputs/*/ 2>/dev/null | head -40

## 2. Install pinned dependencies (keep Colab's CUDA torch)
The universal lock is filtered so Colab's preinstalled CUDA build of torch (and its nvidia/triton libs) is untouched. If pip reports that numpy or another core package changed, use **Runtime → Restart session** and continue from the next cell.

In [ ]:
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!pip install -q --no-deps -e .
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 3. Data
Upload the challenge zip (`Dataset for Challenge-20261101.zip`). The dev split is fetched from GLUE RTE validation.

In [ ]:
%cd /content/rte-challenge
if not (os.path.exists("data/train.jsonl") and os.path.exists("data/dev.jsonl")):
    from google.colab import files
    up = files.upload()  # choose the challenge dataset zip
    ZIP = next(iter(up))
    !python scripts/prepare_data.py --zip "$ZIP" --out data
!python -m pytest -q
!wc -l data/train.jsonl data/dev.jsonl

## 4. Baseline: DeBERTa-v3-base, 5 seeds
Each seed early-stops on a premise-grouped 10% hold-out of train; dev is scored once per seed.

In [ ]:
!python -m rte.train --config configs/baseline_deberta_v3_base.yaml

## 5. Teacher logits (out-of-fold, 5 folds grouped by premise)

In [ ]:
!python -m rte.train --config configs/baseline_deberta_v3_base.yaml --oof

## 6. Download artifacts (optional)
Everything is already in `MyDrive/rte-challenge/outputs`; you can download that folder from Drive instead. Unzip into the local project root. The checkpoint (~0.7 GB) is optional: keep `INCLUDE_MODEL = False` if you only need metrics and teacher logits.

In [ ]:
INCLUDE_MODEL = False
excl = "" if INCLUDE_MODEL else "-x 'outputs/baseline_deberta_v3_base/model/*'"
!zip -q -r teacher_artifacts.zip outputs/baseline_deberta_v3_base outputs/baseline_deberta_v3_base_oof {excl}
!du -h teacher_artifacts.zip
from google.colab import files
files.download("teacher_artifacts.zip")